# Embedding：从编号到向量

> 前两节完成了 Tokenizer：文本被切成 token，每个 token 分配了一个整数编号。但编号只是编号——7 和 3 之间没有大小关系，模型无法从编号判断两个词是否相似。
>
> 这一节引入 Embedding：把离散的 token ID 映射到一个连续的向量空间。我们从「为什么需要向量」出发，逐步建立对词向量的直觉，最后组装出训练中实际使用的 Embedding 层。

Tokenizer 把文本变成了整数序列，比如 [5, 1, 3]。这些整数只是编号——编号本身不携带语义信息。模型看到编号 5 和编号 1，无法判断它们对应的词是否相关。

Embedding 的目标是给每个 token ID 配一个固定长度的实数向量，使得语义相近的词在向量空间中距离更近。但向量不会凭空拥有语义——要理解这些向量从何而来，需要先回答一个问题：为什么用多个数值联合描述一个 token，比用一个编号更好。

In [1]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch 版本: {torch.__version__}")

PyTorch 版本: 2.11.0


## 1. 从编号到向量

用一个数字描述一个东西——比如只用身高——显然不够。168cm 的人有很多，他们各不相同。如果同时用多个特征来描述，情况就不一样了。

| | 尺寸 | 毛茸茸 | 与人亲近 | 独立性 |
|:---|:---|:---|:---|:---|
| cat | 2 | 8 | 6 | 9 |
| dog | 5 | 9 | 9 | 3 |
| algorithm | 0 | 0 | 0 | 0 |

每一行就是一组数值，可以写成一个向量：

```text
cat       → [2, 8, 6, 9]
dog       → [5, 9, 9, 3]
algorithm → [0, 0, 0, 0]
```

维度越多，描述越细。cat 和 dog 在每个维度上都比较接近，algorithm 和它们完全不一样。Embedding 做的事情本质上是相同的——只不过这些特征不是人工挑选的，而是模型在训练过程中自己学出来的。用多个连续的数值联合描述一个 token，数值本身从数据中学来——这就是 Embedding 的核心思想。

### 从颜色 RGB 到词向量

上面用四个维度描述了 cat、dog 和 algorithm，效果比一个编号好得多。这其实就是 Embedding 的核心思路：用多个数值来描述一个东西，而不是只给它一个编号。

但有一个问题：cat 的「毛茸茸」凭什么打 8 分？「与人亲近」的衡量标准是什么？在实际的 Embedding 中，这些数值不是人定的，而是模型自己学出来的。要理解模型是怎么学的，可以先看一个生活中的例子。

颜色有两种描述方式。一种是为每种颜色起一个名字：钴蓝、胭脂红、珊瑚橙……名字越多，需要的词就越多，而且光看名字无法判断两种颜色有多接近。另一种是用 RGB 三个数值来描述，比如 (201, 23, 30)。三个数字就够了，而且判断相似性很直观——两个颜色的数值越接近，看起来就越像。(201, 23, 30) 和 (180, 20, 40) 都是红色系，(23, 180, 201) 是蓝色系，算一下数值的距离就行。

词的表示也面临同样的问题。给每个词一个编号（ID=5 是 cat，ID=12 是 dog），就像给颜色起名字——编号之间没有大小关系，也无法表达「cat 和 dog 很接近，但和 algorithm 差很远」。但如果每个词都有一个类似 RGB 的向量，词与词之间的相似性就可以直接用数值距离来衡量。

这种用多个数值描述单词的方式，叫做分布式表示（distributed representation）。和 one-hot 不同——one-hot 是一个超长的向量，10000 个词就需要 10000 维，其中只有 1 个位置是 1，其余全是 0。分布式表示只用几百维，每一维都是实数，向量之间的距离直接反映语义的远近。

但向量里的数值怎么来？不能随便填，需要从数据中学。要理解学习的过程，需要先搞清楚一件事：上下文如何决定单词的含义。

### 上下文决定含义

用向量表示单词的研究有很多。如果仔细审视这些研究，会发现几乎所有重要方法都基于一个简单的想法：某个单词的含义由它周围的单词形成。

这个想法的含义很直接。单词本身没有含义，单词的含义由它所在的上下文（语境）形成。含义相似的单词经常出现在相似的语境中。比如：

```text
I drink beer.    We drink wine.
I guzzle beer.   We guzzle wine.
```

drink 的附近常有饮料出现，guzzle 的附近也常有饮料出现——drink 和 guzzle 的上下文相似。基于这个观察，可以推断出 guzzle 和 drink 是近义词（guzzle 意为「大口喝」）。

这里所说的上下文，是指某个关注词周围的单词。上下文的大小（即周围单词的数量）称为窗口大小（window size）。窗口大小为 1，上下文包含左右各 1 个单词；窗口大小为 2，上下文包含左右各 2 个单词，以此类推。

```text
语料: You say goodbye and I say hello.

窗口大小为 2，关注词为 goodbye 时：
  You say goodbye and I say hello.
      ←──── 上下文 ────→

  goodbye 的上下文词 = {You, say, and, I}（左侧 2 个 + 右侧 2 个）
```

本节只处理左右单词数量相同、不考虑句子分隔符的情况。

基于这个观察，最直接的做法是统计每个词的上下文中出现了哪些词、各出现多少次，把结果汇总成一张矩阵——称为共现矩阵（co-occurrence matrix）。矩阵的每一行就是一个词的分布式表示：每一维表示对应单词在该词上下文中出现的次数。

共现矩阵能直观地展示「上下文相似 → 向量相似」这一思想。但它有两个明显的局限：矩阵大小是词表大小的平方（词表 10 万就是 10 万×10 万），且每个维度只是原始计数，无法捕捉更复杂的语义关系。

现代 LLM 不再使用这种基于计数的表示，而是通过训练让 Embedding 层自动学到低维密集向量。接下来我们就来看 Embedding 层的具体实现。

## 2. Embedding 查表

`nn.Embedding` 本质是一张矩阵，行数等于词表大小，列数等于向量维度：

```text
矩阵形状: [vocab_size, embed_dim]

第 0 行 → token 0 的向量
第 1 行 → token 1 的向量
第 2 行 → token 2 的向量
...
```

给 Embedding 层一个 token ID，它就取出对应那一行。这些向量一开始是随机的，训练过程中模型会不断调整它们。训练之后，经常出现在相似上下文里的词，向量会靠得更近。

In [ ]:
# 模拟一个 mini 词表，Embedding = vocab_size × embed_dim 的矩阵
vocab = ["the", "cat", "sat", "on", "mat", "dog", "log"]
vocab_size = len(vocab)
embed_dim = 4

embedding = nn.Embedding(vocab_size, embed_dim)

print(f"词表大小: {vocab_size}, Embedding 维度: {embed_dim}")
print(f"Embedding 权重形状: {embedding.weight.shape}  ← 就是一个 {vocab_size}×{embed_dim} 矩阵")
print(f"\n前 3 行初始值（随机）:\n{embedding.weight[:3]}")

In [ ]:
# 查表：给一组 token ID，取出对应的向量
sentence_ids = torch.tensor([0, 1, 2, 3, 0, 4])  # "the cat sat on the mat"
vectors = embedding(sentence_ids)                  # 查表 → [6, 4]

print(f"token IDs: {sentence_ids.tolist()}  →  {[vocab[i] for i in sentence_ids.tolist()]}")
print(f"输出形状: {vectors.shape}  ← [{len(sentence_ids)} 个 token, 每个 {embed_dim} 维]")
print()

# 逐个看
for i, (tid, vec) in enumerate(zip(sentence_ids.tolist(), vectors)):
    print(f"  位置 {i}: '{vocab[tid]}' (ID={tid}) → {vec.tolist()}")

# 关键观察：位置 0 和 4 都是 'the'，向量完全相同
# → 同一个 token 不管出现在哪个位置，查出的向量都一样
print(f"\n关键观察：位置 0 和 4 都是 token 'the'，查出的向量完全相同")
print(f"→ 同一个 token 不管出现在哪个位置，Embedding 查出来的向量都一样")
print(f"→ 模型还无法区分词的顺序——这是下一节位置编码要解决的问题")

**Embedding 是怎么训练出来的**

向量不会凭空拥有语义——它需要从数据中学。业界有两种主要做法：

第一种是**预训练词向量**，以 Word2Vec 和 GloVe 为代表。思路是单独训练 Embedding，然后把它当作固定输入喂给下游模型。比如 Word2Vec 的 Skip-gram 方法：给定一个词，预测它周围可能出现哪些词。通过这个任务，模型被迫学会「哪些词经常出现在相似语境中」，从而让这些词的向量靠近。

第二种是**端到端训练**，也是现代 LLM 采用的方式。Embedding 矩阵作为模型的一部分参数，和 Transformer 的其他层一起，通过反向传播统一更新：

```text
初始化: Embedding 矩阵 E ← 随机值

训练循环:
  for (input_ids, target_ids) in data:
      vectors = E[input_ids]                    # 查表取向量
      logits  = Transformer(vectors)            # 模型前向传播
      loss    = CrossEntropy(logits, targets)   # 计算损失
      loss.backward()                           # 反向传播
      E       = E - lr × E.grad                 # 更新 Embedding 矩阵
```

经过足够多的训练步数，Embedding 矩阵中的每一行就会逐渐学出有意义的向量表示。后面实现 Mini-GPT 时会看到完整的训练流程。

## 3. 工业界的 Embedding 训练实践

前面的 nn.Embedding 查表把概念讲清楚了。但在真实的大语言模型训练中，还有几个工程决策直接影响参数效率和训练稳定性。下面逐一来看。

**权重共享（Weight Tying）**

模型的开头和结尾各有一个大矩阵：输入端的 Embedding 把 token ID 变成向量，输出端的 lm_head 把向量变回词表大小的 logits。两个矩阵形状相同，都是 [vocab_size, d_model]——前者查表取向量，后者把向量投影回词表空间做概率预测。

既然形状相同，能不能共用同一组参数？可以。这种做法称为权重共享（weight tying）：输入 Embedding 和输出投影指向同一块内存，查表和投影用的是同一张矩阵。

以 LLaMA 7B 为例：vocab_size=32000，d_model=4096，不共享要多约 1.3 亿参数。GPT-2、GPT-3、LLaMA、DeepSeek 等都采用了这个技巧。

**Embedding 不做权重衰减**

L2 正则化（weight decay）限制权重的绝对值，防止过拟合。但 Embedding 的每一行代表一个 token 的语义向量——它需要在训练中自由移动到合适的位置。施加 L2 惩罚等于把所有向量往原点拽，干扰语义学习。

业界惯例：Embedding、LayerNorm 的 weight/bias、所有 bias 项都不参与 weight decay。训练时把参数分成两组，一组正常衰减，一组衰减系数为零。

**初始化：更小的标准差**

nn.Embedding 默认用 N(0,1) 初始化。实践中通常用更小的标准差——例如 N(0, 1/√d_model)——避免初始值过大，导致训练早期的梯度不稳定。

**混合精度下的 Embedding**

用 FP16/BF16 训练时，Embedding 矩阵通常保留一份 FP32 副本（master weights）。前向传播转成低精度以加速计算，反向更新时在 FP32 下累积梯度。这是为了避免 FP16 动态范围不足导致的梯度下溢。

下面用代码把这几个实践串起来，模拟一个真实训练循环中 Embedding 的配置方式。

In [ ]:
# === 权重共享（Weight Tying）===
# 输入 Embedding 和输出 lm_head 共享权重——这是 GPT/LLaMA 的标准做法
vocab_size, d_model = 10000, 512

class GPTStyleModel(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)   # token → vector
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)  # vector → logits
        # ★ 关键：让 lm_head 复用 wte 的权重矩阵
        self.lm_head.weight = self.wte.weight

    def forward(self, input_ids):
        x = self.wte(input_ids)
        return self.lm_head(x)

model = GPTStyleModel(vocab_size, d_model)

# 验证：两个 weight 指向同一块内存
print("=" * 50)
print("1. 权重共享验证")
print(f"   wte.weight 内存地址:       {model.wte.weight.data_ptr()}")
print(f"   lm_head.weight 内存地址:   {model.lm_head.weight.data_ptr()}")
print(f"   是同一块内存:              {model.wte.weight.data_ptr() == model.lm_head.weight.data_ptr()}")
print(f"   节省参数量:                {vocab_size * d_model:,} → {vocab_size * d_model * 4 / 1e6:.1f} MB (fp32)")

# 反向传播验证：同一块内存上的梯度正确累加
input_ids = torch.randint(0, vocab_size, (2, 16))
logits = model(input_ids)
logits.mean().backward()
print(f"   两个 grad 指向同一内存:    {model.wte.weight.grad.data_ptr() == model.lm_head.weight.grad.data_ptr()}")

# --- 不共享 vs 共享：参数量对比 ---
class NoTieModel(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        return self.lm_head(self.wte(input_ids))

tie_params = sum(p.numel() for p in GPTStyleModel(vocab_size, d_model).parameters())
no_tie_params = sum(p.numel() for p in NoTieModel(vocab_size, d_model).parameters())
print(f"\n2. 参数量对比 (vocab={vocab_size:,}, d_model={d_model}):")
print(f"   共享权重: {tie_params:,}")
print(f"   独立权重: {no_tie_params:,}")
print(f"   节省:     {no_tie_params - tie_params:,} ({100 * (no_tie_params - tie_params) / no_tie_params:.1f}%)")

for name, vs, dm in [("GPT-2 small", 50257, 768), ("LLaMA 7B", 32000, 4096)]:
    saved = vs * dm
    print(f"   {name}: vocab={vs:,}, d_model={dm} → 节省 {saved:,} 参数 ({saved * 4 / 1e6:.1f} MB)")

In [ ]:
# === 参数分组：Embedding 不做 weight decay ===
# L2 正则化限制权重的绝对值，防止过拟合。但 Embedding 的每一行
# 代表一个 token 的语义向量——它需要在训练中自由移动到合适的位置。
# 施加 L2 惩罚等于把所有向量往原点拽，干扰语义学习。
# 业界惯例：Embedding、LayerNorm、所有 bias 都不做 weight decay。

def make_param_groups(model, weight_decay=0.1):
    """把参数分成两组：一组做 decay，一组不做"""
    decay_params, no_decay_params = [], []
    no_decay_keywords = ['wte', 'norm', 'bias']
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(kw in name.lower() for kw in no_decay_keywords):
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    
    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ]

model2 = GPTStyleModel(vocab_size, d_model)
param_groups = make_param_groups(model2, weight_decay=0.1)

decay_count = sum(p.numel() for p in param_groups[0]['params'])
no_decay_count = sum(p.numel() for p in param_groups[1]['params'])
print(f"参数分组（weight_decay=0.1）:")
print(f"   做衰减的参数:   {decay_count:,}")
print(f"   不做衰减的参数: {no_decay_count:,}  ← Embedding 在这里")

optimizer = torch.optim.AdamW(param_groups, lr=1e-3)
for i, pg in enumerate(optimizer.param_groups):
    wd = pg['weight_decay']
    count = sum(p.numel() for p in pg['params'])
    print(f"   Group {i}: {count:,} params, weight_decay={wd}")

# === 初始化：更小的标准差 ===
# nn.Embedding 默认用 N(0,1) 初始化，实践中通常用更小的标准差
# 避免 Embedding 向量的初始值过大，导致训练早期梯度不稳定
print(f"\n初始化对比:")
torch.manual_seed(42)
default_emb = nn.Embedding(100, 16)
custom_emb = nn.Embedding(100, 16)
nn.init.normal_(custom_emb.weight, mean=0.0, std=1.0 / math.sqrt(d_model))

print(f"   nn.Embedding 默认 N(0,1):       std = {default_emb.weight.std().item():.4f}")
print(f"   自定义 N(0, 1/√d_model):         std = {custom_emb.weight.std().item():.4f}")
print(f"   目标 1/√{d_model} = {1.0 / math.sqrt(d_model):.4f}")
print(f"   → 更小的初始值 → 训练早期梯度更稳定")

In [ ]:
# === 模拟一次训练 step（含梯度裁剪）===
model3 = GPTStyleModel(vocab_size, d_model)
optimizer = torch.optim.AdamW(
    make_param_groups(model3, weight_decay=0.1), lr=1e-3
)

batch = torch.randint(0, vocab_size, (4, 32))   # 随机 batch
targets = torch.randint(0, vocab_size, (4, 32))  # 随机 targets

logits = model3(batch)                            # 前向
loss = nn.functional.cross_entropy(
    logits.view(-1, vocab_size), targets.view(-1)
)
loss.backward()                                   # 反向
torch.nn.utils.clip_grad_norm_(model3.parameters(), 1.0)  # 梯度裁剪
optimizer.step()                                  # 更新参数
optimizer.zero_grad()

print(f"模拟训练 step:")
print(f"   loss: {loss.item():.4f}")
print(f"   wte.weight 已更新: {model3.wte.weight.grad is None} → 梯度已清零")
print(f"   → 完整流程：forward → loss → backward → clip → step → zero_grad")


## 小结

这一节所学的内容：

- Token ID 只是编号，不能直接作为模型的数值输入
- Embedding 是一张 [vocab_size, d_model] 的矩阵，给定 ID 就取出对应行的向量
- 单词的含义由上下文决定——含义相似的词出现在相似的语境中
- 共现矩阵是基于计数的最简分布式表示，现代 LLM 改用可训练的低维密集向量
- nn.Embedding 是可学习的查表，训练后语义相近的词在向量空间中距离更近
- 工业界实践：权重共享节省参数、Embedding 不做 weight decay、更小的初始化标准差

同一个 token 不管出现在哪个位置，查出的向量都相同——模型还需要知道每个 token 在句子中的位置。下一节引入位置编码来解决这个问题。

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：Embedding 查表**

Token ID 本身没有语义，Embedding 会把 ID 查成向量。

小提示：`embedding_table[token_ids]` 可以一次取多行。

In [ ]:
# 作业 1：Embedding 查表填空
import torch

embedding_table = torch.tensor([
    [1.0, 0.0],  # token 0
    [0.0, 1.0],  # token 1
    [1.0, 1.0],  # token 2
])
token_ids = torch.tensor([2, 0, 1])

# TODO：把下面三引号里的内容替换成你的代码
vectors = """在这里根据 token_ids 从 embedding_table 里取出对应向量"""

assert not isinstance(vectors, str), "请先替换三引号里的占位内容"
expected = torch.tensor([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
assert torch.equal(vectors, expected), vectors
print("✅ 作业 1 通过：你记住了 Embedding 的核心就是查表")

## 参考资料

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017 — Transformer 原始论文，Embedding 乘以 √d_model 的惯例来自此文
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — 对原始论文的逐行实现，`Embeddings` 类中可以看到 `self.lut(x) * math.sqrt(self.d_model)` 的写法
- Mikolov et al., [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781), 2013 — Word2Vec，分布式表示的经典工作